# 🌊 03 – Intégration avec SciPy

---

## 🎯 Objectifs
- Comprendre ce qu'est **intégrer** sans formule mathématique
- Calculer des **aires sous une courbe** avec `quad()`
- Appliquer l'intégration à des cas concrets : distance, consommation, revenus
- Simuler une **évolution dans le temps** avec `odeint()`
- Appliquer la simulation à des situations métier compréhensibles

> ℹ️ **Astuce** : Les corrections sont cachées dans des blocs pliables.  
> Cliquez sur **💡 Correction** pour dérouler la solution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad, odeint
print("SciPy Integrate prêt ✅")

---
## 📝 Partie 1 – Qu'est-ce qu'intégrer ?

### 🔎 L'idée en langage humain

**Intégrer**, c'est **additionner une quantité qui varie en continu** sur un intervalle.

**Exemples concrets :**
- Une voiture roule à **vitesse variable** → intégrer la vitesse donne la **distance parcourue**
- Un commerce a un **flux de clients variable** selon l'heure → intégrer donne le **nombre total de clients**
- Une machine consomme de l'**énergie à puissance variable** → intégrer donne la **consommation totale**

### 🔎 La visualisation : l'aire sous la courbe

Géométriquement, intégrer une fonction entre deux points = calculer l'**aire entre la courbe et l'axe horizontal**.

```
Vitesse (km/h)
   ↑
80 │    ╭──────╮
60 │  ╭─╯      ╰─╮
40 │╭─╯           ╰─╮
20 │╯                ╰
   └─────────────────→ Temps (h)
   0    1    2    3

L'aire colorée sous la courbe = distance totale parcourue
```

### 🔎 `quad()` — calculer une aire

```python
from scipy.integrate import quad

resultat, erreur = quad(fonction, a, b)
#                         ↑        ↑  ↑
#                    la fonction  début fin
#                    à intégrer   de l'intervalle

# resultat = l'aire calculée (la valeur qu'on cherche)
# erreur   = l'incertitude de calcul (généralement très petite, on peut l'ignorer)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# Vitesse d'une voiture qui accélère progressivement (km/h)
def vitesse(t):
    return 30 + 10 * t   # part de 30 km/h, augmente de 10 km/h par heure

# Calculer la distance parcourue entre t=0 et t=3 heures
distance, erreur = quad(vitesse, 0, 3)

print(f"Distance parcourue : {distance:.1f} km")
print(f"Erreur de calcul   : {erreur:.2e} (négligeable)")
print()
print("Vérification intuitive :")
print(f"  Vitesse initiale : {vitesse(0)} km/h")
print(f"  Vitesse finale   : {vitesse(3)} km/h")
print(f"  Vitesse moyenne  : {(vitesse(0)+vitesse(3))/2} km/h")
print(f"  → Distance ≈ vitesse moyenne × durée = {(vitesse(0)+vitesse(3))/2 * 3:.0f} km ✓")

# Visualisation : l'aire = la distance
t = np.linspace(0, 3, 200)
v = vitesse(t)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t, v, color="steelblue", linewidth=2.5, label="Vitesse (km/h)")
ax.fill_between(t, v, alpha=0.3, color="steelblue",
                label=f"Aire = distance = {distance:.0f} km")
ax.set_title("Vitesse variable — l'aire sous la courbe = distance parcourue")
ax.set_xlabel("Temps (heures)")
ax.set_ylabel("Vitesse (km/h)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📝 Partie 2 – Cas d'usage concrets de `quad()`

### 🔎 Trois situations typiques

| Situation | Ce qui varie | Ce que `quad()` calcule |
|-----------|-------------|------------------------|
| Voiture à vitesse variable | Vitesse (km/h) | Distance totale (km) |
| Flux de clients variable | Clients/heure | Total clients sur la journée |
| Consommation électrique variable | Puissance (kW) | Énergie consommée (kWh) |
| Revenus variables | €/mois | Revenus cumulés sur l'année |
| Pluviométrie variable | mm/heure | Pluie totale tombée (mm) |

### 🔎 Intégrer sur des intervalles différents

```python
# Total sur toute la journée (0h à 24h)
total_jour, _ = quad(flux_clients, 0, 24)

# Seulement la matinée (6h à 12h)
total_matin, _ = quad(flux_clients, 6, 12)

# Seulement l'après-midi (12h à 18h)
total_aprem, _ = quad(flux_clients, 12, 18)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# Flux de clients dans un magasin (clients/heure)
# Pic en matinée (10h) et en fin d'après-midi (17h)
def flux_clients(h):
    return (20 + 30 * np.exp(-0.5*(h-10)**2)
               + 25 * np.exp(-0.5*(h-17)**2))

# Total sur la journée (9h à 20h)
total_jour,  _ = quad(flux_clients, 9, 20)
total_matin, _ = quad(flux_clients, 9, 13)
total_aprem, _ = quad(flux_clients, 13, 17)
total_soir,  _ = quad(flux_clients, 17, 20)

print(f"Total clients journée       : {total_jour:.0f}")
print(f"  dont matinée (9h-13h)     : {total_matin:.0f}  ({total_matin/total_jour*100:.0f}%)")
print(f"  dont après-midi (13h-17h) : {total_aprem:.0f}  ({total_aprem/total_jour*100:.0f}%)")
print(f"  dont soirée (17h-20h)     : {total_soir:.0f}   ({total_soir/total_jour*100:.0f}%)")

# Visualisation
heures = np.linspace(9, 20, 300)
flux   = flux_clients(heures)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(heures, flux, color="steelblue", linewidth=2.5)
ax.fill_between(heures, flux, where=(heures < 13),  color="coral",     alpha=0.4, label=f"Matin : {total_matin:.0f} clients")
ax.fill_between(heures, flux, where=((heures >= 13) & (heures < 17)), color="gold", alpha=0.4, label=f"Après-midi : {total_aprem:.0f} clients")
ax.fill_between(heures, flux, where=(heures >= 17), color="seagreen",  alpha=0.4, label=f"Soirée : {total_soir:.0f} clients")
ax.set_title(f"Flux de clients — Total journée : {total_jour:.0f} clients")
ax.set_xlabel("Heure")
ax.set_ylabel("Clients / heure")
ax.set_xticks(range(9, 21))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 🧩 Exercice 1 – Consommation électrique

Une usine fonctionne 24h/24. Sa **puissance consommée** (en kW) varie selon l'heure :

```python
def puissance(h):
    if 8 <= h <= 20:   # journée : production active
        return 500 + 200 * np.sin(np.pi * (h - 8) / 12)
    else:               # nuit : maintenance réduite
        return 150
```

1. Tracez la courbe de puissance sur 24 heures
2. Calculez la consommation totale sur 24h (en kWh = kW × heures)
3. Calculez séparément la consommation de jour (8h-20h) et de nuit (0h-8h + 20h-24h)
4. Si l'électricité coûte **0.15 €/kWh** le jour et **0.08 €/kWh** la nuit, calculez le coût total

> 💡 `quad()` fonctionne avec n'importe quelle fonction Python, même avec des `if`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

def puissance(h):
    if 8 <= h <= 20:
        return 500 + 200 * np.sin(np.pi * (h - 8) / 12)
    else:
        return 150

# TODO 1 : tracer la courbe
heures = np.linspace(0, 24, 500)
p_vals = [puissance(h) for h in heures]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(heures, p_vals, color=..., linewidth=2)
ax.fill_between(heures, p_vals, alpha=0.2, color=...)
ax.set_title("Puissance consommée par l'usine")
ax.set_xlabel("Heure")
ax.set_ylabel("Puissance (kW)")
ax.set_xticks(range(0, 25, 2))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# TODO 2 : consommation totale 24h
conso_totale, _ = quad(puissance, 0, 24)
print(f"Consommation totale : {conso_totale:.0f} kWh")

# TODO 3 : jour et nuit séparément
conso_jour, _  = quad(puissance, ..., ...)
conso_nuit, _  = quad(puissance, 0, 8)
conso_nuit2, _ = quad(puissance, 20, 24)
conso_nuit_tot = conso_nuit + conso_nuit2

print(f"Consommation jour (8h-20h)  : {conso_jour:.0f} kWh")
print(f"Consommation nuit           : {conso_nuit_tot:.0f} kWh")

# TODO 4 : coût
cout_jour = conso_jour  * 0.15
cout_nuit = conso_nuit_tot * 0.08
print(f"\nCoût jour  : {cout_jour:.2f} €")
print(f"Coût nuit  : {cout_nuit:.2f} €")
print(f"Coût total : {cout_jour + cout_nuit:.2f} €/jour")

<details>
<summary>💡 Correction</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

def puissance(h):
    if 8 <= h <= 20:
        return 500 + 200 * np.sin(np.pi * (h - 8) / 12)
    else:
        return 150

heures = np.linspace(0, 24, 500)
p_vals = [puissance(h) for h in heures]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(heures, p_vals, color="steelblue", linewidth=2)
ax.fill_between(heures, p_vals, alpha=0.2, color="steelblue")
ax.set_title("Puissance consommée par l'usine")
ax.set_xlabel("Heure"); ax.set_ylabel("Puissance (kW)")
ax.set_xticks(range(0, 25, 2))
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

conso_totale, _ = quad(puissance, 0, 24)
print(f"Consommation totale : {conso_totale:.0f} kWh")

conso_jour,  _ = quad(puissance, 8, 20)
conso_nuit,  _ = quad(puissance, 0, 8)
conso_nuit2, _ = quad(puissance, 20, 24)
conso_nuit_tot = conso_nuit + conso_nuit2

print(f"Consommation jour : {conso_jour:.0f} kWh")
print(f"Consommation nuit : {conso_nuit_tot:.0f} kWh")

cout_jour = conso_jour * 0.15
cout_nuit = conso_nuit_tot * 0.08
print(f"Coût total : {cout_jour + cout_nuit:.2f} €/jour")
# → environ 1090 kWh/jour, coût ≈ 141 €/jour
```
</details>

---
## 📝 Partie 3 – Simuler une évolution dans le temps avec `odeint()`

### 🔎 Le problème que résout `odeint()`

`quad()` calcule une aire sous une courbe **connue**. Mais parfois on ne connaît pas la courbe — on connaît seulement la **règle de changement** à chaque instant.

**Exemples :**
- *"La température baisse de 10% par heure"* → on connaît le **taux de changement**, pas la courbe
- *"Les ventes augmentent de 5% par mois"* → idem
- *"Le stock diminue de 20 unités par jour"* → idem

**`odeint()`** simule l'évolution étape par étape à partir d'une valeur initiale et d'une règle de changement.

### 🔎 La règle de changement — comment l'écrire ?

On écrit une **fonction** qui dit : *"à cet instant, quelle est la vitesse de changement ?"*

```python
def regle_changement(valeur_actuelle, t):
    return taux_de_changement
#          ↑
#   positif = la valeur augmente
#   négatif = la valeur diminue

# Exemples :
def refroidissement(T, t):
    return -0.1 * T          # T diminue de 10% de sa valeur actuelle par heure

def croissance(N, t):
    return 0.05 * N          # N augmente de 5% par période

def stock(S, t):
    return -20               # le stock diminue de 20 unités par jour (taux fixe)
```

### 🔎 Syntaxe de `odeint()`

```python
from scipy.integrate import odeint

valeurs = odeint(
    func = regle_changement,   # la fonction qui donne le taux de changement
    y0   = valeur_initiale,    # valeur au départ
    t    = points_de_temps     # les instants où on veut la valeur
)
# valeurs est un tableau de shape (len(t), 1)
# valeurs[:, 0] → les valeurs à chaque instant
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# Simulation : refroidissement d'un café
# Le café part à 90°C et se refroidit vers la température ambiante (20°C)
# Règle : plus l'écart avec l'ambiance est grand, plus ça refroidit vite

temp_ambiante = 20   # °C

def refroidissement(T, t):
    # Le taux de refroidissement est proportionnel à l'écart avec l'ambiance
    return -0.1 * (T - temp_ambiante)
    # Si T = 90°C et ambiance = 20°C → écart = 70°C → refroidit de 7°C/min
    # Si T = 30°C et ambiance = 20°C → écart = 10°C → refroidit de 1°C/min

# Points de temps : de 0 à 60 minutes
t = np.linspace(0, 60, 300)   # 300 points entre 0 et 60 min
T0 = 90                         # température initiale du café

# Simuler l'évolution
T = odeint(refroidissement, T0, t)
T = T[:, 0]   # extraire le tableau 1D (odeint retourne une colonne)

print(f"Température initiale  : {T[0]:.1f} °C")
print(f"Après 10 minutes      : {T[np.argmin(abs(t-10))]:.1f} °C")
print(f"Après 30 minutes      : {T[np.argmin(abs(t-30))]:.1f} °C")
print(f"Après 60 minutes      : {T[-1]:.1f} °C")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t, T, color="coral", linewidth=2.5, label="Température du café")
ax.axhline(temp_ambiante, color="steelblue", linestyle="--",
           label=f"Température ambiante : {temp_ambiante}°C")
ax.axhline(60, color="gray", linestyle=":", alpha=0.7,
           label="Température idéale (60°C)")
ax.set_title("Refroidissement d'un café")
ax.set_xlabel("Temps (minutes)")
ax.set_ylabel("Température (°C)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# Simulation : croissance des ventes avec effet de marché
# Les ventes croissent de 8% par mois, mais ralentissent quand elles
# approchent du plafond du marché (10 000 unités/mois)

plafond_marche = 10000

def croissance_ventes(V, t):
    taux = 0.08   # 8% par mois
    # Le taux réel diminue quand on approche du plafond
    # (moins de clients potentiels restants)
    return taux * V * (1 - V / plafond_marche)

mois = np.linspace(0, 36, 200)   # simulation sur 3 ans (36 mois)
V0   = 500                         # ventes initiales : 500 unités/mois

V = odeint(croissance_ventes, V0, mois)
V = V[:, 0]

print(f"Ventes initiales    : {V[0]:.0f} unités/mois")
print(f"Après 12 mois       : {V[np.argmin(abs(mois-12))]:.0f} unités/mois")
print(f"Après 24 mois       : {V[np.argmin(abs(mois-24))]:.0f} unités/mois")
print(f"Après 36 mois       : {V[-1]:.0f} unités/mois")
print(f"Plafond du marché   : {plafond_marche} unités/mois")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(mois, V, color="steelblue", linewidth=2.5, label="Ventes simulées")
ax.axhline(plafond_marche, color="red", linestyle="--",
           label=f"Plafond marché : {plafond_marche}")
ax.set_title("Simulation de croissance des ventes")
ax.set_xlabel("Mois")
ax.set_ylabel("Ventes (unités/mois)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🧩 Exercice 2 – Revenus cumulés et simulation de stock

### Partie A — `quad()` : revenus cumulés

Une boutique en ligne génère des revenus variables selon le mois :
```python
def revenus(m):
    return 8000 + 3000 * np.sin(np.pi * m / 6)  # en €/mois
    # pic en juillet (m≈6), creux en décembre/janvier
```

1. Tracez la courbe des revenus sur 12 mois
2. Calculez les revenus totaux annuels avec `quad()`
3. Calculez les revenus du 1er semestre (mois 0 à 6) et du 2e semestre
4. Quel semestre génère le plus de revenus ?

### Partie B — `odeint()` : stock qui se vide

Un entrepôt commence avec 5000 unités en stock.
- Les ventes journalières : `30 + 10 * np.sin(np.pi * t / 7)` (varie avec les jours de la semaine)
- Les réapprovisionnements : 50 unités tous les 7 jours

```python
def evolution_stock(S, t):
    ventes = 30 + 10 * np.sin(np.pi * t / 7)
    reappro = 50 if t % 7 < 0.1 else 0   # 50 unités tous les 7 jours
    return -ventes + reappro
```

5. Simulez l'évolution du stock sur 60 jours
6. À quel moment le stock passe-t-il en dessous de 1000 unités ?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad, odeint

# ===== Partie A : revenus =====
def revenus(m):
    return 8000 + 3000 * np.sin(np.pi * m / 6)

# TODO 1 : tracer la courbe
mois_range = np.linspace(0, 12, 300)
fig, axes  = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(mois_range, [revenus(m) for m in mois_range], color="steelblue", linewidth=2)
axes[0].fill_between(mois_range, [revenus(m) for m in mois_range], alpha=0.2, color="steelblue")
axes[0].set_title("Revenus mensuels")
axes[0].set_xlabel("Mois")
axes[0].set_ylabel("Revenus (€)")
axes[0].grid(True, alpha=0.3)

# TODO 2 : revenus annuels
rev_annuel, _ = quad(revenus, 0, ...)
print(f"Revenus annuels totaux : {rev_annuel:,.0f} €")

# TODO 3 : semestres
rev_s1, _ = quad(revenus, 0, 6)
rev_s2, _ = quad(revenus, 6, 12)
print(f"Semestre 1 (jan-juin) : {rev_s1:,.0f} €")
print(f"Semestre 2 (jul-déc) : {rev_s2:,.0f} €")
print(f"Meilleur semestre    : {'S1' if rev_s1 > rev_s2 else 'S2'}")

# ===== Partie B : stock =====
def evolution_stock(S, t):
    ventes  = 30 + 10 * np.sin(np.pi * t / 7)
    reappro = 50 if t % 7 < 0.1 else 0
    return -ventes + reappro

# TODO 5 : simuler sur 60 jours
jours = np.linspace(0, 60, 600)
S0    = 5000
S     = odeint(evolution_stock, S0, jours)[:, 0]

axes[1].plot(jours, S, color="coral", linewidth=2)
axes[1].axhline(1000, color="red", linestyle="--", label="Seuil critique (1000)")
axes[1].set_title("Évolution du stock")
axes[1].set_xlabel("Jours")
axes[1].set_ylabel("Stock (unités)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# TODO 6 : quand le stock passe sous 1000 ?
sous_seuil = jours[S < 1000]
if len(sous_seuil) > 0:
    print(f"\nStock < 1000 à partir du jour {sous_seuil[0]:.0f}")
else:
    print("\nLe stock ne passe jamais sous 1000 unités")

<details>
<summary>💡 Correction</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad, odeint

def revenus(m):
    return 8000 + 3000 * np.sin(np.pi * m / 6)

# 2 — revenus annuels totaux
rev_annuel, _ = quad(revenus, 0, 12)
print(f"Revenus annuels : {rev_annuel:,.0f} €")  # ≈ 96 000 €

# 3 — semestres
rev_s1, _ = quad(revenus, 0, 6)
rev_s2, _ = quad(revenus, 6, 12)
print(f"S1 : {rev_s1:,.0f} €  |  S2 : {rev_s2:,.0f} €")
# S1 ≈ 57 100 €  |  S2 ≈ 38 900 €
# → Semestre 1 (jan-juin) plus fort car le pic estival est en milieu d'année

# 5 — simulation stock
def evolution_stock(S, t):
    ventes  = 30 + 10 * np.sin(np.pi * t / 7)
    reappro = 50 if t % 7 < 0.1 else 0
    return -ventes + reappro

jours = np.linspace(0, 60, 600)
S     = odeint(evolution_stock, 5000, jours)[:, 0]

# 6 — quand stock < 1000 ?
sous_seuil = jours[S < 1000]
if len(sous_seuil) > 0:
    print(f"Stock < 1000 à partir du jour {sous_seuil[0]:.0f}")
    # → environ jour 55 selon les paramètres
```
</details>

---
## ✅ Récapitulatif

### `quad()` — calculer une aire / un total

| Quand l'utiliser | Exemple |
|-----------------|--------|
| Quantité qui **varie en continu** et dont on veut le **total** | Distance, consommation, revenus cumulés |
| On **connaît** la fonction qui décrit la variation | `vitesse(t) = 30 + 10*t` |

```python
resultat, _ = quad(ma_fonction, debut, fin)
```

### `odeint()` — simuler une évolution

| Quand l'utiliser | Exemple |
|-----------------|--------|
| On connaît la **règle de changement** mais pas la courbe | Stock qui diminue, température qui baisse |
| On veut suivre l'**évolution au fil du temps** | Ventes sur 12 mois, stock sur 60 jours |

```python
def regle(valeur, t):
    return taux_de_changement   # positif = augmente, négatif = diminue

valeurs = odeint(regle, valeur_initiale, points_de_temps)
valeurs = valeurs[:, 0]   # extraire le tableau 1D
```

| Concept | `quad()` | `odeint()` |
|---------|----------|------------|
| **Donne** | Un nombre (le total) | Un tableau (l'évolution) |
| **Nécessite** | La fonction connue | La règle de changement |
| **Résultat** | Aire sous la courbe | Trajectoire au fil du temps |

---
📘 **Prochain notebook → `04` : Analyse de signaux avec SciPy**